# Capstone (ML-11): Which Pages Should You Refresh First?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sohamrathi17/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

The capstone notebook. It mirrors my deployed paper, [Which Pages Should You Refresh First?](https://sohamrathi17.github.io/flyrank-ml-internship/): question, data, method, results with a sealed final month and bootstrap intervals, limitations, recommendations, and the ML-12 closing cells.

## 1. Question

**Which pages should a content editor review first, so that the pages most likely to keep losing search visibility next month are near the top of each client's list?**

- **Decision supported:** which 10-20 pages, out of hundreds per client, an editor opens this month, and whether to refresh, expand, monitor or leave each one alone.
- **Unit:** one page at a monthly decision moment, ranked inside its own client.
- **Output:** a ranked review queue per client with a reason code, an action and a confidence label.
- **Outcome measured:** next month's average daily Search Console impressions more than 20% below this month's.
- **Success, fixed in ML-03 before any model:** precision@20 per client (share of true decliners in the top 20), plus the share of next month's lost impressions those 20 pages hold. Both are compared with the base rate and with a transparent rule.

Lane: **Refresh / Content Opportunity Scoring** (Lane 2).

In [1]:
import os, sys, subprocess, json, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
# numpy 2 + Apple Accelerate prints spurious "overflow in matmul" warnings inside sklearn; results are unaffected
warnings.filterwarnings("ignore", category=RuntimeWarning, module="sklearn")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sohamrathi17/flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
else:
    while not os.path.isdir("work/scripts") and os.getcwd() != "/":
        os.chdir("..")
sys.path.insert(0, "work/scripts")

import numpy as np, pandas as pd, sklearn
import lane_data as L

receipts = {name: json.load(open(f"work/outputs/{name}.json"))
            for name in ["baseline_metrics", "model_metrics", "validation_metrics", "paper_data"]}
print("Receipts loaded:", list(receipts))
print(f"scikit-learn {sklearn.__version__} | seed {L.SEED}")

Receipts loaded: ['baseline_metrics', 'model_metrics', 'validation_metrics', 'paper_data']
scikit-learn 1.6.1 | seed 42


## 2. Data

- **Release:** FlyRank ML Internship warehouse on Hugging Face (`FlyRank/internship-warehouse`, gated, pseudonymised).
- **Tables:** `fact_content_daily_performance` (one row per page per day, monthly partitions) and `dim_content` (creation date and keyword metadata only).
- **Windows:** four monthly decision moments, each with features from one month and the label from the next:

| Role | Features | Label | Used for |
|---|---|---|---|
| time-aware training | Feb 2026 | Mar 2026 | ML-09, ML-10 |
| development | Mar 2026 | Apr 2026 | ML-04 to ML-10 (grouped cross-validation) |
| sealed training | Apr 2026 | May 2026 | fitting the frozen recipe for the sealed test |
| **sealed test** | May 2026 | **Jun 2026** | evaluated **once**, after everything else was frozen |

- **Slice:** pages with Search Console data on every day of both months (filtered with `gsc_data_available IS TRUE`) and at least 100 impressions in the feature month.
- **Excluded, and why:**
  - every column from the label month, because it is the answer
  - `content_updated_date` and `last_optimized_date`, which are snapshot dates and postdate the decision moment for 78% of pages
  - GA4 engagement columns, available on only about 4% of rows
  - all IDs as features (grouping and splitting only)
- **Public-safety:** IDs are pseudonymous hashes and are never printed, and no URLs or queries are used. Only aggregate numbers leave this notebook.

In [2]:
slices = {}
for fm, lm in [("2026-02", "2026-03"), ("2026-03", "2026-04"), ("2026-04", "2026-05"), ("2026-05", "2026-06")]:
    fr = L.build_frame(fm, lm)
    slices[f"{fm} -> {lm}"] = {"pages": len(fr), "clients": fr["client_hash_id"].nunique(),
                               "declining (pooled)": round(fr["is_declining"].mean(), 3)}
pd.DataFrame(slices).T

,pages,clients,declining (pooled)
2026-02 -> 2026-03,43596.0,23.0,0.211
2026-03 -> 2026-04,50299.0,32.0,0.474
2026-04 -> 2026-05,62056.0,35.0,0.525
2026-05 -> 2026-06,58399.0,37.0,0.594


## 3. Methodology

- **Assumption:** a slide that is already visible inside a month, together with a few page signals, says something about the next month. I make no causal assumption.
- **Features (16, all knowable at the decision moment):**
  - **Feature-month search signals:** impressions, clicks, CTR, impression-weighted position, momentum (late-half vs early-half daily impressions), size of the within-month drop, days with impressions, daily volatility, position change, biggest-day share.
  - **Page age.**
  - **Keyword metadata:** search volume, competition, keyword length, content type, intent.
- **Label:** next-month daily impressions below 80% of this month's.
- **Baseline (ML-07, frozen):** score = impressions per day lost between the first and second half of the month, ×1.5 when CTR is below its position-tier median. Built after two signal checks, CTR-vs-position (CONFIRMED) and momentum (CONFIRMED).
- **Model (ML-08):** logistic regression. The ranking score = model probability × (1 + within-month drop per day). Gradient-boosted trees were tried and not chosen: they had better ROC-AUC but a weaker top 20.
- **Validation:**
  - grouped 5-fold cross-validation by client on the development month
  - a time-aware check (train February→March, test March→April)
  - one sealed final-month test (train April→May, test May→June)
  - a client-level bootstrap for uncertainty
- **Leakage checks (ML-04, ML-09):**
  - a deliberate leak (next-month impressions) sends ROC-AUC to 0.999, so the harness catches it
  - no label or snapshot column is in the features
  - removing the snapshot-based keyword metadata barely changes the results
  - the population filter that uses outcome-month coverage is disclosed (it drops 18.5% of eligible pages)

In [3]:
print("Model features:", L.MODEL_FEATURES)
assert not set(L.NEVER_FEATURES) & set(L.MODEL_FEATURES)
v = receipts["validation_metrics"]
print(f"Deliberate-leak ROC-AUC: {v['leak_test_auc']}  (honest grouped ROC-AUC: {v['designs']['after: grouped by client']['ROC-AUC']})")
print(f"Pages dropped by the full-coverage filter: {v['population_dropped_share']:.1%}")

Model features: ['log_imp', 'avg_position', 'ctr', 'age_days', 'momentum', 'log_clicks', 'days_with_imp', 'imp_cv', 'position_change', 'top_day_share', 'log_search_volume', 'has_search_volume', 'competition', 'keyword_token_count', 'log_drop', 'content_type', 'main_intent']
Deliberate-leak ROC-AUC: 0.9994  (honest grouped ROC-AUC: 0.6327)
Pages dropped by the full-coverage filter: 18.5%


## 4. Results (vs baseline)

**4a. Development month (March → April), grouped by client, out-of-fold.** From the ML-08 receipt:

In [4]:
dev = pd.DataFrame(receipts["model_metrics"]["results"]).T
dev.round(3)

,precision@20,loss captured@20,ROC-AUC
random order (expected),0.489,0.104,NaN
ML-07 baseline rule,0.711,0.241,NaN
logistic regression,0.750,0.197,0.633
boosted trees (classifier),0.734,0.174,0.653
boosted trees (expected loss),0.691,0.264,NaN
logistic prob x March drop/day,0.718,0.242,NaN


**4b. Across evaluations: rule vs model on the same pages each time,** including the sealed final month. The sealed test reruns the frozen recipe (`work/scripts/sealed_test.py`). Its receipt `work/outputs/sealed_test_metrics.json` was written by the single first run, and the cell below re-computes it and checks it still matches.

In [5]:
import importlib, sealed_test
sealed_receipt = json.load(open("work/outputs/sealed_test_metrics.json"))
sealed_now = sealed_test.run()
assert sealed_now["model_combined"] == sealed_receipt["model_combined"] and sealed_now["ml07_rule"] == sealed_receipt["ml07_rule"], \
    "sealed result changed: the recipe or data is not the frozen one"

vd = receipts["validation_metrics"]["designs"]
rows = [
    {"evaluation": "Mar->Apr, grouped CV",       "rule p@20": vd["ML-07 rule (no training)"]["precision@20"], "model p@20": vd["after: grouped by client"]["precision@20"],
     "rule loss@20": vd["ML-07 rule (no training)"]["loss_captured@20"], "model loss@20": vd["after: grouped by client"]["loss_captured@20"], "base rate": vd["after: grouped by client"]["base_rate"]},
    {"evaluation": "Mar->Apr, trained on Feb->Mar", "rule p@20": vd["ML-07 rule (no training)"]["precision@20"], "model p@20": vd["after: time-aware (Feb->Mar train)"]["precision@20"],
     "rule loss@20": vd["ML-07 rule (no training)"]["loss_captured@20"], "model loss@20": vd["after: time-aware (Feb->Mar train)"]["loss_captured@20"], "base rate": vd["after: grouped by client"]["base_rate"]},
    {"evaluation": "SEALED May->Jun, trained on Apr->May", "rule p@20": sealed_receipt["ml07_rule"]["precision@20"], "model p@20": sealed_receipt["model_combined"]["precision@20"],
     "rule loss@20": sealed_receipt["ml07_rule"]["loss_captured@20"], "model loss@20": sealed_receipt["model_combined"]["loss_captured@20"], "base rate": sealed_receipt["model_combined"]["base_rate"]},
]
across = pd.DataFrame(rows).set_index("evaluation")
print(f"Sealed test: {sealed_receipt['test']['pages']:,} pages, {sealed_receipt['model_combined']['clients']} evaluated clients, ROC-AUC {sealed_receipt['roc_auc']}")
across.round(3)

Sealed test: 58,399 pages, 28 evaluated clients, ROC-AUC 0.6677


,rule p@20,model p@20,rule loss@20,model loss@20,base rate
evaluation,,,,,
"Mar->Apr, grouped CV",0.711,0.718,0.241,0.242,0.489
"Mar->Apr, trained on Feb->Mar",0.711,0.720,0.241,0.235,0.489
"SEALED May->Jun, trained on Apr->May",0.720,0.732,0.356,0.340,0.479


**4c. Uncertainty: client-level bootstrap.** The metric is an average over 22-28 clients, so I resample *clients* (2,000 times, seed 42) and look at the 95% interval of model minus rule.

In [6]:
def per_client_pair(frame, model_score):
    a = L.per_client_metrics(frame, model_score).set_index("client")
    b = L.per_client_metrics(frame, L.baseline_score(frame)).set_index("client")
    return pd.DataFrame({"dp": a["precision_at_k"] - b["precision_at_k"],
                         "dl": a["loss_captured_at_k"] - b["loss_captured_at_k"],
                         "rule_p": b["precision_at_k"], "base": b["base_rate"]})

def boot(diff, n=2000, seed=L.SEED):
    rng = np.random.default_rng(seed)
    means = [diff.iloc[rng.integers(0, len(diff), len(diff))].mean() for _ in range(n)]
    return diff.mean(), *np.percentile(means, [2.5, 97.5])

# development month, grouped OOF
dev_frame = L.add_drop_feature(L.build_frame("2026-03", "2026-04"))
p_dev = L.out_of_fold(dev_frame, L.make_logistic, "is_declining")
pair_dev = per_client_pair(dev_frame, p_dev * (1 + L.drop_per_day(dev_frame)))

# sealed month (frozen recipe)
tr = L.add_drop_feature(L.build_frame("2026-04", "2026-05"))
te = L.add_drop_feature(L.build_frame("2026-05", "2026-06"))
p_seal = L.make_logistic().fit(tr[L.MODEL_FEATURES], tr["is_declining"]).predict_proba(te[L.MODEL_FEATURES])[:, 1]
pair_seal = per_client_pair(te, p_seal * (1 + L.drop_per_day(te)))

uncert = {}
for name, pr in [("dev Mar->Apr", pair_dev), ("sealed May->Jun", pair_seal)]:
    uncert[name] = {
        "model - rule, precision@20": boot(pr["dp"]),
        "model - rule, loss captured@20": boot(pr["dl"]),
        "rule - base rate, precision@20": boot(pr["rule_p"] - pr["base"]),
    }
    print(f"{name} ({len(pr)} clients):")
    for k, (m, lo, hi) in uncert[name].items():
        print(f"   {k:32} {m:+.3f}   95% CI [{lo:+.3f}, {hi:+.3f}]")

dev Mar->Apr (22 clients):
   model - rule, precision@20       +0.007   95% CI [-0.007, +0.021]
   model - rule, loss captured@20   +0.001   95% CI [-0.004, +0.006]
   rule - base rate, precision@20   +0.223   95% CI [+0.159, +0.284]
sealed May->Jun (28 clients):
   model - rule, precision@20       +0.012   95% CI [-0.002, +0.027]
   model - rule, loss captured@20   -0.016   95% CI [-0.030, -0.004]
   rule - base rate, precision@20   +0.241   95% CI [+0.188, +0.301]


**Reading the results:**

- **The transparent rule is the main result.** Ranking pages by how much they already slid inside the month puts about 0.71-0.72 true decliners in each client's top 20 against a base rate of about 0.48-0.49, and the bootstrap interval for "rule minus base rate" stays well above zero in both the development and the sealed month. Its top 20 also holds about 2.3× the share of lost impressions that a random top 20 would (0.241 vs 0.104 in the development month).
- **The learned model adds very little on top.** It is a point or so higher on precision@20 in every evaluation, including the sealed month, but the 95% intervals for "model minus rule" on precision include zero in both periods. On lost impressions captured the two are tied in the development month, and in the sealed month the rule is measurably *better* (by about 1.6 points, interval entirely below zero). I cannot claim the model beats the rule.
- **Both held up on a month nobody tuned on.** The sealed May→June test was run once, after everything was frozen, and landed close to the development numbers, even though the portfolio's decline rate moved a lot between months (21% → 47% → 59% pooled).
- **What the model *does* add** is a readable set of weights that confirm why the rule works: a large drop and high impressions with few clicks lean toward decline, while clicks and momentum lean toward holding. Its probability can also feed the confidence label. In practice, I would ship the rule's ordering, with the model kept as a monthly check.

## 5. Limitations

- **Observational, not causal.** Nothing here says a refresh *causes* recovery. The queue shows where a slide is visible and tends to continue.
- **The model does not beat the rule** within measurable uncertainty. The paper says so plainly.
- **Probabilities do not transfer between months** (ML-09: trained on a 21% month, predicted about 25% on a 47% month). Only the *ranking* is used.
- **Population filter uses the outcome month:** pages without full next-month Search Console coverage are dropped (18.5% of eligible pages in March). Pages that vanished were never scored.
- **Few, unequal clients:**
  - only 22-28 clients have 20+ pages, and the three largest hold about two thirds of the pages
  - clients with fewer than ~100 pages see no lift
  - only 8 clients were completely new to the time-aware model
- **One label definition** (-20% month over month). Seasonality and demand shifts look the same as a page-level decline.
- **Spike-then-recover pages** are the most common false alarm. The rule cannot tell a fading spike from a real slide.
- **Snapshot metadata** (`dim_content`) reflects the table's later build date. The update dates are excluded, and the keyword fields were tested and found not to drive results.

## 6. Ranked recommendations

From the ML-10 playbook (the queue for April, built on 31 March from a model trained on February→March):

1. **Start each month with the top 10-20 of each client's queue**, high-confidence rows first. In April, 79% of high-confidence top-20 picks did decline, against 60% of low-confidence ones and a base rate near 49%.
2. **Refresh** sliding page-one pages, **expand** sliding positions 11-20, and **monitor** slides that look like a one-day spike or sit deep in the results.
3. **Before any edit, check the page's daily impressions** in Search Console. A fading spike, a seasonal topic or a tracking gap means *monitor*.
4. **Do not use it for clients under ~100 pages.** Review those by hand.
5. **Show ranks and reasons, never probabilities.** Re-check precision@20 against the rule every month.

In [7]:
pdq = receipts["paper_data"]
print("Top-20 picks graded on April, by confidence:", {r["label"]: (r["n"], r["rate"]) for r in pdq["top20_by_confidence"]})
print("By action:", {r["label"]: (r["n"], r["rate"]) for r in pdq["top20_by_action"]})
pd.DataFrame(pdq["example_queue"]).set_index("rank_in_client")

Top-20 picks graded on April, by confidence: {'high': (215, 0.786), 'medium': (104, 0.7212), 'low': (121, 0.6033)}
By action: {'expand': (45, 0.7111), 'monitor': (142, 0.7113), 'protect': (2, 0.5), 'refresh': (251, 0.7291)}


,action,confidence,reason_codes,position
rank_in_client,,,,
1,monitor,high,daily impressions fell 64% within March,40.4
2,refresh,high,daily impressions fell 50% within March; page ...,6.9
3,monitor,low,daily impressions fell 52% within March; 30% o...,2.7
4,monitor,high,daily impressions fell 60% within March; CTR b...,28.1
5,monitor,high,daily impressions fell 39% within March,27.0
6,monitor,high,daily impressions fell 44% within March; CTR b...,28.0
7,monitor,low,daily impressions fell 77% within March; 58% o...,2.0
8,refresh,high,daily impressions fell 23% within March; CTR b...,2.9
9,refresh,high,daily impressions fell 44% within March; page ...,8.5


## 7. Artifacts the paper embeds

Everything the paper shows comes from committed receipts in `work/outputs/` (`baseline_metrics.json`, `model_metrics.json`, `validation_metrics.json`, `paper_data.json`, `sealed_test_metrics.json`) plus the capstone metrics written below. `work/scripts/build_paper.py` injects these numbers into the page (`docs/index.html`), so the charts cannot drift from the notebooks.

In [8]:
def r(x): return round(float(x), 4)
capstone = {
    "sealed_test": sealed_receipt,
    "across_evaluations": {k: {c: r(v) for c, v in row.items()} for k, row in across.to_dict("index").items()},
    "bootstrap_95ci": {k: {m: {"mean": r(v[0]), "lo": r(v[1]), "hi": r(v[2])} for m, v in d.items()} for k, d in uncert.items()},
    "bootstrap": {"resamples": 2000, "unit": "client", "seed": L.SEED},
}
with open("work/outputs/capstone_metrics.json", "w") as fh:
    json.dump(capstone, fh, indent=2)
subprocess.run([sys.executable, "work/scripts/build_paper.py"], check=True)
print("Wrote work/outputs/capstone_metrics.json and rebuilt docs/index.html")

paper build: docs/index.html (46 KB) from 6 receipts
Wrote work/outputs/capstone_metrics.json and rebuilt docs/index.html


## ML-12: demo outline, social post, employer summary

**5-minute demo outline**
1. **(0:00-0:45) The problem.** An editor has 567 pages for a typical client and time for 20. Which 20? Show how "biggest pages first" and "oldest first" fail (ML-03).
2. **(0:45-1:45) The data and the honest setup.** Daily Search Console data; features from one month, outcome from the next. Show the deliberate leak jumping to 0.999 and coming back down (ML-04).
3. **(1:45-2:45) The rule.** "How much did the page already slide this month?" Precision@20 0.71 vs 0.49 base rate. Walk two top-10 rows, including one miss (a fading spike).
4. **(2:45-3:45) The model, and the honest verdict.** Grouped by client, time-aware, a sealed month run once. The model is about 1 point better, and the bootstrap says that is not distinguishable from zero. The rule ships.
5. **(3:45-5:00) The playbook.** One client's queue with actions, reasons and confidence; high-confidence rows declined 79% of the time. The no-go list and the monthly checks. Close on the limits: observational, ranking not probabilities.

**Social-post cut**
> Which pages should you refresh first? I tested this on ~50,000 pages from real Search Console data, ranking each site's pages and checking against the following month. A plain rule, "how much has this page already slid this month?", put 71% true decliners in each site's top 20 vs a 49% base rate. A machine-learning model on 16 signals added about 1 point, within noise, and both held up on a final month I only touched once. Sometimes the honest result is: keep the simple rule, and measure it. Full paper + notebooks: [link]

**3-sentence employer summary**
> I built and validated a decision-support ranking that tells content editors which pages to review first, using ~79 million rows of pseudonymised daily Google Search Console data from the FlyRank ML Internship warehouse. I designed a leakage-safe monthly setup (features from one month, outcome from the next), compared a transparent rule against logistic regression and gradient boosting with client-grouped cross-validation, a time-aware split, a sealed final month and a client-level bootstrap. The rule reached 0.71-0.72 precision@20 against a ~0.49 base rate and held up on the sealed month; the models added no significant gain, and I shipped the rule with a monitored action playbook and published the full write-up.

## Quality checks

Confirmed for this notebook:

- [x] Every section above is filled: markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (all outputs shown are from a full run)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to the repo under `work/notebooks/`
- [x] My deployed paper has **all 9 sections**, including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 is in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.